FEATURE EXTRACTION 


In [1]:
!pip install efficientnet-pytorch -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.9 MB/s eta 0:00:00


In [2]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from efficientnet_pytorch import EfficientNet
import torchvision.transforms as transforms

In [3]:
# --- Phase 2: Load Model (DEFINITIVE FIX) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = EfficientNet.from_pretrained('efficientnet-b0')
model._fc = torch.nn.Identity() # This is the critical line to get 1280 features
model = model.to(device)
model.eval()
feature_extractor = model # The model itself is the extractor

Using device: cuda


Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth
100%|██████████| 20.4M/20.4M [00:00<00:00, 177MB/s]


Loaded pretrained weights for efficientnet-b0


In [4]:
# --- Phase 3 & 4: Helper Functions (DEFINITIVE FIX) ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_frames(video_path, frame_rate=1):
    frames = []
    try:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps == 0: return np.array([])
        frame_interval = int(fps / frame_rate) if fps >= frame_rate else 1
        count = 0
        success, frame = cap.read()
        while success:
            if count % frame_interval == 0:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
            success, frame = cap.read()
            count += 1
    finally:
        if 'cap' in locals() and cap.isOpened():
            cap.release()
    return np.array(frames)

In [5]:
def get_features(frames, batch_size=32):
    features = []
    with torch.no_grad():
        for i in range(0, len(frames), batch_size):
            batch = frames[i:i+batch_size]
            batch = torch.stack([transform(f).to(device) for f in batch])
            output = feature_extractor(batch) # No .squeeze() needed
            features.append(output.cpu().numpy())
    return np.vstack(features)

In [6]:
# ===================================================================
# PHASE 5: PROCESS ENTIRE DATASET
# ===================================================================
# This is the main loop that processes all your video files.

# --- IMPORTANT: Update this path to your main dataset directory ---
data_dir = "/kaggle/input/dataset" 
save_dir = "/kaggle/working/features/"
os.makedirs(save_dir, exist_ok=True)

print("\nStarting feature extraction process...")

# Define your folder structure
anomaly_parent_folders = ["Anomaly-Videos-Part-1", "Anomaly-Videos-Part-2"]
normal_folder = "Normal-Videos-Part-1"

# Process Anomaly Videos
print("\nProcessing Anomaly videos...")
for parent_folder in anomaly_parent_folders:
    parent_path = os.path.join(data_dir, parent_folder)
    if not os.path.exists(parent_path):
        print(f"Warning: Directory not found - {parent_path}")
        continue
    
    for anomaly_type in sorted(os.listdir(parent_path)):
        folder_path = os.path.join(parent_path, anomaly_type)
        if not os.path.isdir(folder_path):
            continue

        for video in tqdm(os.listdir(folder_path), desc=f"Processing {anomaly_type}"):
            video_path = os.path.join(folder_path, video)
            output_filename = f"Anomaly_{anomaly_type}_{video.split('.')[0]}.npy"
            output_path = os.path.join(save_dir, output_filename)

            if os.path.exists(output_path):
                continue

            try:
                frames = extract_frames(video_path, frame_rate=1)
                if len(frames) > 0:
                    feats = get_features(frames)
                    np.save(output_path, feats)
            except Exception as e:
                print(f"Error processing {video_path}: {e}")

# Process Normal Videos
print("\nProcessing Normal videos...")
normal_path = os.path.join(data_dir, normal_folder)
if os.path.exists(normal_path):
    for video in tqdm(os.listdir(normal_path), desc="Processing Normal"):
        video_path = os.path.join(normal_path, video)
        output_filename = f"Normal_{video.split('.')[0]}.npy"
        output_path = os.path.join(save_dir, output_filename)

        if os.path.exists(output_path):
            continue
            
        try:
            frames = extract_frames(video_path, frame_rate=1)
            if len(frames) > 0:
                feats = get_features(frames)
                np.save(output_path, feats)
        except Exception as e:
            print(f"Error processing {video_path}: {e}")

print("\n\nFeature extraction complete! Your features are saved in the 'features/' directory.")


Starting feature extraction process...

Processing Anomaly videos...


Processing Fighting: 100%|██████████| 50/50 [01:19<00:00,  1.59s/it]



Processing Normal videos...


Processing Normal: 100%|██████████| 150/150 [03:13<00:00,  1.29s/it]



Feature extraction complete! Your features are saved in the 'features/' directory.


In [7]:
import numpy as np
import os

# --- Sanity Check ---
# Path to your saved features
features_dir = "/kaggle/working/features/"
# Get a list of a few feature files to check
feature_files = os.listdir(features_dir)[:5]

for file_name in feature_files:
    file_path = os.path.join(features_dir, file_name)
    
    # Load the feature vector
    features = np.load(file_path)
    
    print(f"--- Checking file: {file_name} ---")
    # 1. Check the shape: Should be (Number of Frames, 1280)
    print(f"Shape: {features.shape}")
    
    # 2. Check for NaN (Not-a-Number) values
    has_nan = np.isnan(features).any()
    print(f"Contains NaN values: {has_nan}")

    # 3. Check the mean value (should not be zero)
    mean_value = features.mean()
    print(f"Mean feature value: {mean_value:.4f}\n")

--- Checking file: Anomaly_Arson_Arson010_x264.npy ---
Shape: (106, 1280)
Contains NaN values: False
Mean feature value: 0.0668

--- Checking file: Anomaly_Assault_Assault018_x264.npy ---
Shape: (13, 1280)
Contains NaN values: False
Mean feature value: 0.1064

--- Checking file: Anomaly_Arson_Arson030_x264.npy ---
Shape: (169, 1280)
Contains NaN values: False
Mean feature value: 0.0761

--- Checking file: Normal_Normal_Videos_641_x264.npy ---
Shape: (120, 1280)
Contains NaN values: False
Mean feature value: 0.0421

--- Checking file: Anomaly_Arrest_Arrest020_x264.npy ---
Shape: (99, 1280)
Contains NaN values: False
Mean feature value: 0.1333



In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# --- t-SNE Visualization ---
# Select a few representative videos of different, clear classes
files_to_visualize = {
    'Arson': '/kaggle/working/Anomaly_Arson_Arson009_x264.npy',
    'Explosion': '/kaggle/working/Anomaly_Explosion_Explosion001_x264.npy',
    'Normal': '/kaggle/working/Normal_Normal-Videos-Part-1_Normal_Videos_003_x264.npy'
}

all_features = []
all_labels = []

print("Loading features for visualization...")
for label, path in files_to_visualize.items():
    if os.path.exists(path):
        feats = np.load(path)
        all_features.append(feats)
        all_labels.extend([label] * len(feats))
    else:
        print(f"Warning: File not found for visualization: {path}")

if not all_features:
    print("No features loaded. Skipping visualization.")
else:
    # Combine all features into one big array
    X = np.vstack(all_features)

    print("Running t-SNE... (this may take a minute)")
    tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300)
    tsne_results = tsne.fit_transform(X)

    # Plot the results
    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        x=tsne_results[:,0], y=tsne_results[:,1],
        hue=all_labels,
        palette=sns.color_palette("hls", len(files_to_visualize)),
        legend="full",
        alpha=0.7
    )
    plt.title('t-SNE Visualization of Video Frame Features')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.show()

Loading features for visualization...
No features loaded. Skipping visualization.


In [9]:
normal_count = len([f for f in os.listdir(features_dir) if f.startswith('Normal')])
anomaly_count = len([f for f in os.listdir(features_dir) if f.startswith('Anomaly')])
print(f"Found {normal_count} Normal videos and {anomaly_count} Anomaly videos.")

Found 150 Normal videos and 400 Anomaly videos.
